# **Import**

In [1]:
import numpy as np
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# **Data**

In [2]:
destination_rating = pd.read_csv('data/tourism_rating.csv')
destination = pd.read_csv('data/tourism_with_id.csv')
user = pd.read_csv('data/user.csv')

In [3]:
destination = destination[destination['City']=='Jakarta']
destination = destination.drop(destination.columns[[11, 12]], axis=1)
destination = destination.drop(destination.columns[[7]], axis=1)
destination_rating = pd.merge(destination_rating, destination[['Place_Id']], how='right', on='Place_Id')
user = pd.merge(user, destination_rating[['User_Id']], how='right', on='User_Id').drop_duplicates().sort_values('User_Id')

# ***Univariate Exploratory Data Analysis*** **(UEDA)**

In [4]:
destination.describe().apply(lambda s: s.apply('{0:.2f}'.format))

,Place_Id,Price,Rating,Lat,Long
count,84.00,84.00,84.00,84.00,84.00
mean,42.50,45130.95,4.49,-6.09,106.78
std,24.39,115657.34,0.20,0.80,0.32
min,1.00,0.00,4.00,-6.38,103.93
25%,21.75,0.00,4.40,-6.21,106.81
50%,42.50,4500.00,4.50,-6.18,106.82
75%,63.25,25000.00,4.60,-6.13,106.84
max,84.00,900000.00,5.00,1.08,106.96


In [5]:
print(f"Terdapat {destination['Place_Name'].nunique()} Tempat Wisata di DKI Jakarta")
print(f"Terdiri dari {destination['Category'].nunique()} Kategori Wisata yaitu")
print('Kategori Wisata  :', destination['Category'].unique())

Terdapat 84 Tempat Wisata di DKI Jakarta
Terdiri dari 6 Kategori Wisata yaitu
Kategori Wisata  : ['Budaya' 'Taman Hiburan' 'Cagar Alam' 'Bahari' 'Pusat Perbelanjaan'
 'Tempat Ibadah']


In [6]:
columns_category_type = destination['Category'].unique().tolist()

for label, count in destination['Category'].value_counts().items():
    print("Jumlah Tempat Wisata dengan Kategori", label, ":", count)

Jumlah Tempat Wisata dengan Kategori Budaya : 32
Jumlah Tempat Wisata dengan Kategori Taman Hiburan : 27
Jumlah Tempat Wisata dengan Kategori Pusat Perbelanjaan : 10
Jumlah Tempat Wisata dengan Kategori Bahari : 8
Jumlah Tempat Wisata dengan Kategori Cagar Alam : 4
Jumlah Tempat Wisata dengan Kategori Tempat Ibadah : 3


In [7]:
destination_rating.describe().apply(lambda s: s.apply('{0:.2f}'.format))

,User_Id,Place_Id,Place_Ratings
count,1920.00,1920.00,1920.00
mean,150.45,42.62,3.01
std,86.93,23.95,1.38
min,1.00,1.00,1.00
25%,74.00,22.75,2.00
50%,150.00,42.00,3.00
75%,224.25,63.00,4.00
max,300.00,84.00,5.00


# ***Multivariate Exploratory Data Analysis*** **(MEDA)**

In [8]:
Kategori_Biaya = destination.groupby('Category').agg({'Price': 'mean'})
Kategori_Biaya = Kategori_Biaya.reset_index()
Kategori_Biaya = Kategori_Biaya.rename(columns={'Price': 'Mean Price'})
Kategori_Biaya = Kategori_Biaya.sort_values(by='Mean Price', ascending=False)

# **Modelling - Content Based Filtering**

TF-IDF

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer()
tf.fit(destination['Category'])
tf.get_feature_names_out()

array(['alam', 'bahari', 'budaya', 'cagar', 'hiburan', 'ibadah',
       'perbelanjaan', 'pusat', 'taman', 'tempat'], dtype=object)

In [10]:
tfidf_matrix = tf.fit_transform(destination['Category'])

In [11]:
tfidf_matrix.todense()

matrix([[0.        , 0.        , 1.        , 0.        , 0.        ,
         0.        , 0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 1.        , 0.        , 0.        ,
         0.        , 0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.70710678,
         0.        , 0.        , 0.        , 0.70710678, 0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.70710678,
         0.        , 0.        , 0.        , 0.70710678, 0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.70710678,
         0.        , 0.        , 0.        , 0.70710678, 0.        ],
        [0.        , 0.        , 0.        , 0.        , 0.70710678,
         0.        , 0.        , 0.        , 0.70710678, 0.        ],
        [0.70710678, 0.        , 0.        , 0.70710678, 0.        ,
         0.        , 0.        , 0.        , 0.        , 0.        ],
        [0.        , 0.    

In [12]:
pd.DataFrame(
    tfidf_matrix.todense(),
    columns=tf.get_feature_names_out(),
    index=destination.Place_Name
).sample(10, axis=0)

,alam,bahari,budaya,cagar,hiburan,ibadah,perbelanjaan,pusat,taman,tempat
Place_Name,,,,,,,,,,
Pelabuhan Marina,0.000000,1.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0
Museum Layang-layang,0.000000,0.0,1.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0
Freedom Library,0.000000,0.0,1.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0
Museum Basoeki Abdullah,0.000000,0.0,1.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0
Waterboom PIK (Pantai Indah Kapuk),0.000000,0.0,0.0,0.000000,0.707107,0.0,0.0,0.0,0.707107,0.0
Atlantis Water Adventure,0.000000,0.0,0.0,0.000000,0.707107,0.0,0.0,0.0,0.707107,0.0
Istana Negara Republik Indonesia,0.000000,0.0,1.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0
Pulau Pari,0.000000,1.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0
Museum Wayang,0.000000,0.0,1.0,0.000000,0.000000,0.0,0.0,0.0,0.000000,0.0


Cosine Similarity

In [13]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix)

In [14]:
cosine_sim_df = pd.DataFrame(
    cosine_sim, index=destination.Place_Name, columns=destination.Place_Name)

Implementasi Sistem Recommendasi

In [15]:
def destination_recommendations(place_name, similarity_data=cosine_sim_df, items=destination[['Place_Name', 'Category', 'Rating', 'Price']], k=10):
    index = similarity_data.loc[:,place_name].to_numpy().argpartition(range(-1, -k, -1))
    closest = similarity_data.columns[index[-1:-(k+2):-1]]
    closest = closest.drop(place_name, errors='ignore')
    return pd.DataFrame(closest).merge(items).head(k)

In [16]:
jakarta = destination[destination['City'] == 'Jakarta']
# jakarta = jakarta[['Place_Id','Place_Name','Category','Rating','Price']]
jakarta

,Place_Id,Place_Name,Description,Category,City,Price,Rating,Coordinate,Lat,Long
0,1,Monumen Nasional,Monumen Nasional atau yang populer disingkat d...,Budaya,Jakarta,20000,4.6,"{'lat': -6.1753924, 'lng': 106.8271528}",-6.175392,106.827153
1,2,Kota Tua,"Kota tua di Jakarta, yang juga bernama Kota Tu...",Budaya,Jakarta,0,4.6,"{'lat': -6.137644799999999, 'lng': 106.8171245}",-6.137645,106.817125
2,3,Dunia Fantasi,Dunia Fantasi atau disebut juga Dufan adalah t...,Taman Hiburan,Jakarta,270000,4.6,"{'lat': -6.125312399999999, 'lng': 106.8335377}",-6.125312,106.833538
3,4,Taman Mini Indonesia Indah (TMII),Taman Mini Indonesia Indah merupakan suatu kaw...,Taman Hiburan,Jakarta,10000,4.5,"{'lat': -6.302445899999999, 'lng': 106.8951559}",-6.302446,106.895156
4,5,Atlantis Water Adventure,Atlantis Water Adventure atau dikenal dengan A...,Taman Hiburan,Jakarta,94000,4.5,"{'lat': -6.12419, 'lng': 106.839134}",-6.124190,106.839134
...,...,...,...,...,...,...,...,...,...,...
79,80,Plaza Indonesia,Plaza Indonesia diresmikan pada awal tahun 199...,Pusat Perbelanjaan,Jakarta,0,4.7,"{'lat': -6.193925900000002, 'lng': 106.8222158}",-6.193926,106.822216
80,81,Mall Thamrin City,Thamrin City atau Thamrin City Mall merupakan ...,Pusat Perbelanjaan,Jakarta,0,4.4,"{'lat': -6.1946096, 'lng': 106.817905}",-6.194610,106.817905
81,82,Museum Satria Mandala,Museum Satria Mandala adalah museum sejarah pe...,Budaya,Jakarta,5000,4.5,"{'lat': -6.231698499999999, 'lng': 106.8185425}",-6.231699,106.818543
82,83,Alive Museum Ancol,Museum kini tidak hanya menawarkan benda – ben...,Taman Hiburan,Jakarta,200000,4.3,"{'lat': -6.117534399999999, 'lng': 106.857313}",-6.117534,106.857313


In [19]:
# ten_place = input('Masukkan Nama Tempat')
# destination_recommendations(ten_place)

destination_recommendations('Kidzania')

,Place_Name,Category,Rating,Price
0,The Escape Hunt,Taman Hiburan,4.4,70000
1,Wisata Agro Edukatif Istana Susu Cibugary,Taman Hiburan,4.5,35000
2,Jakarta Aquarium dan Safari,Taman Hiburan,4.6,185000
3,Taman Legenda Keong Emas,Taman Hiburan,4.5,30000
4,Taman Menteng,Taman Hiburan,4.5,0
5,Taman Suropati,Taman Hiburan,4.6,0
6,Skyrink - Mall Taman Anggrek,Taman Hiburan,4.5,110000
7,Sea World,Taman Hiburan,4.5,115000
8,Taman Lapangan Banteng,Taman Hiburan,4.7,0
9,Taman Situ Lembang,Taman Hiburan,4.5,0
